# WA-LIME study — full pipeline

Generates every number for the paper from scratch: dataset descriptives, model
performance, explanations, explanation stability, and rank/SNR analysis.

**Run order:** top to bottom. Cells 1–6 are setup and define everything; each
*Stage* cell below is independent once setup has run, so you can re-run just the
stage you are iterating on.

**Before you start:** put `Full_Dataset.xlsx` next to this notebook (or set
`DATA_PATH` in the config cell), and set `QUICK = True` for a ~3 minute
smoke run before committing to the full one.

Outputs are written to `paper_outputs/` as CSV and also displayed inline.

## 1 · Install and import

In [ ]:
# lime 0.2.0.1 fails to build under setuptools >= 70 (AttributeError: install_layout).
# The runtime code is fine, only the build step breaks, so fall back to copying
# the package out of the sdist.
import importlib, subprocess, sys

def ensure(pkg, imp=None):
    try:
        importlib.import_module(imp or pkg); return
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    try:
        importlib.import_module(imp or pkg)
    except ImportError:
        if (imp or pkg) != "lime":
            raise
        import glob, shutil, site, tarfile, tempfile
        tmp = tempfile.mkdtemp()
        subprocess.run([sys.executable, "-m", "pip", "download", "lime==0.2.0.1",
                        "--no-deps", "--no-binary", ":all:", "-d", tmp], check=True)
        tarfile.open(glob.glob(tmp + "/lime-*.tar.gz")[0]).extractall(tmp)
        dest = site.getsitepackages()[0]
        shutil.copytree(glob.glob(tmp + "/lime-0.2.0.1/lime")[0],
                        dest + "/lime", dirs_exist_ok=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "scikit-image", "tqdm"], check=False)
        importlib.invalidate_caches()

for p, i in [("pandas", None), ("numpy", None), ("scipy", None),
             ("scikit-learn", "sklearn"), ("openpyxl", None), ("lime", "lime")]:
    ensure(p, i)

import itertools, json, os, time, warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 50)
print("imports ok")

## 2 · Config

Every knob for the study. The paper's Methods section should quote these values.
Everything is seeded, so a given config reproduces exactly.

In [ ]:
QUICK = True    # True  -> ~3 min, for checking the notebook runs
                # False -> the real run (~1 h); use this for the paper

DATA_PATH   = "Full_Dataset.xlsx"

SPLIT       = "random"   # "random" = row-level 80/20 (comparable to prior CMAPSS
                         # work); "grouped" = engine-level, no engine on both
                         # sides. Stage 02 reports BOTH regardless; this picks
                         # which model the explanation study runs on.
TEST_SIZE   = 0.20
SPLIT_SEED  = 42
MODEL_SEED  = 0

N_PERTURB     = 5000            # LIME num_samples per explanation
K_LIST        = (5, 8, 10)      # report Jaccard at each k
K_DISPLAY     = 8               # k for the rank-occupancy tables
B_AGG         = 5               # WA-LIME aggregation depth
INSTANCE_SEED = 20260905
RUN_COMPUTE_MATCHED = True      # plain LIME at B_AGG x N_PERTURB perturbations

if QUICK:
    N_INSTANCES, R, B_SWEEP = 2, 5, (2, 5)
else:
    N_INSTANCES, R, B_SWEEP = 8, 100, (2, 3, 5, 10, 20)

# QUICK results go somewhere else on purpose: a smoke run must never overwrite
# the numbers you are about to publish.
OUT_DIR = "paper_outputs_quick" if QUICK else "paper_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"QUICK={QUICK}  ->  writing to {OUT_DIR}/")
print(f"instances={N_INSTANCES}  R={R} repetitions  B_sweep={B_SWEEP}")
if QUICK:
    print("\n*** SMOKE RUN - these numbers are not for the paper. ***")

## 3 · Data

RUL is defined per engine as `last cycle - current cycle`. The cell asserts the
`RUL` column in the file actually matches that definition before anything is
quoted from it.

In [ ]:
def load_frame(path=None):
    """Read the dataset and normalise the exported column names."""
    path = path or DATA_PATH
    if not os.path.exists(path):
        raise SystemExit(f"Dataset not found at {path!r}. Put Full_Dataset.xlsx "
                         f"next to this notebook or set DATA_PATH.")
    df = pd.read_excel(path)
    df.columns = [c.replace("Unit Number", "unit number")
                   .replace("Time (Cycles)", "time") for c in df.columns]
    df.columns = [" ".join(c.split()).lower()
                  if c not in ("unit number", "time", "RUL") else c
                  for c in df.columns]
    return df

def load_xy():
    """Features and target: drop 'unit number', keep time + settings + sensors."""
    df = load_frame()
    return df, df.drop(columns=["unit number"]).drop("RUL", axis=1), df["RUL"].copy()

SHORT = lambda c: (c.replace("sensor measurement ", "S")
                    .replace("operational setting ", "OS"))

df_all, X_all, y_all = load_xy()

# verify the RUL column matches its definition
_rul = df_all.groupby("unit number")["time"].transform("max") - df_all["time"]
assert (_rul == df_all["RUL"]).all(), "RUL column does not match max(time)-time"

print(f"{len(df_all):,} rows | {df_all['unit number'].nunique()} engines | "
      f"{X_all.shape[1]} features | RUL verified")

## 4 · Model

In [ ]:
def build_model(split=SPLIT):
    """Fit the GBRM. Returns (gb, X_train, X_test, y_train, y_test, groups_test)."""
    df, X, y = load_xy()
    groups = df["unit number"].values
    if split == "grouped":
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE,
                                        random_state=SPLIT_SEED).split(X, y, groups))
    elif split == "random":
        tr, te = train_test_split(np.arange(len(X)), test_size=TEST_SIZE,
                                  random_state=SPLIT_SEED)
    else:
        raise ValueError(f"unknown split {split!r}")
    gb = GradientBoostingRegressor(random_state=MODEL_SEED).fit(X.iloc[tr], y.iloc[tr])
    return gb, X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te], groups[te]


def pick_instances(y_test, n=None):
    """Instances to explain, stratified across the RUL range.

    One instance is an anecdote. Spreading them over early/mid/late life makes
    the stability result a property of the model, not of one lucky cycle.
    """
    n = n or N_INSTANCES
    rng = np.random.RandomState(INSTANCE_SEED)
    yv = y_test.values
    edges = np.quantile(yv, np.linspace(0, 1, n + 1))
    seen, out = set(), []
    for lo, hi in zip(edges[:-1], edges[1:]):
        pool = np.where((yv >= lo) & (yv <= hi))[0]
        if len(pool):
            p = int(rng.choice(pool))
            if p not in seen:
                seen.add(p); out.append(p)
    return out

gb, X_train, X_test, y_train, y_test, groups_test = build_model()
COLS = list(X_train.columns)
INSTANCES = pick_instances(y_test)
print(f"split={SPLIT}  train={len(X_train):,}  test={len(X_test):,}")
print(f"instances to explain: {INSTANCES}")

## 5 · Explanation engines

Two things here are deliberate and worth a sentence in the Methods section:

- **`num_features` defaults to all 25.** LIME's `num_features` does feature
  selection and then *refits the surrogate on that subset*, so asking for 8
  changes the model being explained, not just what is displayed. Truncate for
  display, never before computing a metric.
- **`local_exp[1]`, not `[0]`.** In regression mode LIME stores the sign-correct
  weights under key 1 (`Explanation.dummy_label == 1`) and their negation under
  key 0. `as_list()` and `show_in_notebook()` both read key 1.

In [ ]:
def lime_weights(row, seed, n_perturb=N_PERTURB, k=None):
    """One LIME explanation as a dense SIGNED weight vector. Returns (w, R2_local)."""
    ex = LimeTabularExplainer(X_train.values, feature_names=COLS, mode="regression",
                              discretize_continuous=False, random_state=int(seed))
    e = ex.explain_instance(row, gb.predict, num_features=k or len(COLS),
                            num_samples=n_perturb)
    w = np.zeros(len(COLS))
    for i, v in e.local_exp[1]:
        w[i] = v
    return w, float(e.score)


def wa_lime_weights(row, seed, B=B_AGG, signed=False, n_perturb=N_PERTURB):
    """WA-LIME: mean of B independent LIME runs.

    signed=False averages |w| (ranks features); signed=True keeps direction,
    which is the actionable content for a maintenance engineer. Report both.
    """
    acc, scores = np.zeros(len(COLS)), []
    for b in range(B):
        w, sc = lime_weights(row, seed * 1000 + b, n_perturb=n_perturb)
        acc += (w if signed else np.abs(w)) / B
        scores.append(sc)
    return acc, float(np.mean(scores))


class Progress:
    """Minimal progress line so long cells show they are alive."""
    def __init__(self, total, label=""):
        self.total, self.label, self.n, self.t0 = total, label, 0, time.time()
    def step(self, k=1):
        self.n += k
        el = time.time() - self.t0
        eta = el / self.n * (self.total - self.n) if self.n else 0
        print(f"\r  {self.label} {self.n}/{self.total}  "
              f"elapsed {el:5.0f}s  eta {eta:5.0f}s   ", end="", flush=True)
    def done(self):
        print(f"\r  {self.label} {self.total}/{self.total}  "
              f"done in {time.time()-self.t0:.0f}s" + " " * 20)

print("explanation engines ready")

## 6 · Stability metrics

**Spearman is computed over all 25 features, not the intersection of two top-k
sets.** Restricting to the intersection nearly guarantees ρ ≈ 1: the surviving
features are the high-SNR ones whose order is stable by construction, so the
statistic discards exactly the features whose ordering is in doubt.

In [ ]:
def jaccard_topk(a, b, k):
    """|A n B| / |A u B| on the top-k features by |weight|."""
    sa = set(np.argsort(-np.abs(a))[:k])
    sb = set(np.argsort(-np.abs(b))[:k])
    return len(sa & sb) / len(sa | sb)


def spearman_full(a, b):
    """Spearman on the FULL feature ranking by |weight|."""
    return float(spearmanr(np.abs(a), np.abs(b)).statistic)


def pairwise_stability(W, k_list=K_LIST):
    """W: (R, n_features) from R independent repetitions of one method."""
    R = len(W)
    out = {}
    for k in k_list:
        out[f"jaccard@{k}"] = float(np.mean(
            [jaccard_topk(W[i], W[j], k) for i, j in itertools.combinations(range(R), 2)]))
    out["spearman"] = float(np.mean(
        [spearman_full(W[i], W[j]) for i, j in itertools.combinations(range(R), 2)]))
    out["n_pairs"] = R * (R - 1) // 2
    return out


def bootstrap_over_runs(W, k, n_boot=400, seed=0):
    """Resample RUNS, not pairs. Pairs are dependent, so a naive CI over pairs
    is too narrow."""
    rng = np.random.RandomState(seed)
    stats = []
    for _ in range(n_boot):
        idx = rng.choice(len(W), len(W), replace=True)
        v = [jaccard_topk(W[i], W[j], k)
             for i, j in itertools.combinations(idx, 2) if i != j]
        if v:
            stats.append(np.mean(v))
    return float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))


def save(name, obj):
    """Write a DataFrame to paper_outputs/ and return it for display."""
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=False)
    print(f"  -> {p}")
    return obj

print("metrics ready")

---
# Stage 01 · Dataset descriptives

Produces the dataset table and the per-sensor overview.

In [ ]:
life = df_all.groupby("unit number")["time"].max()
t01 = pd.DataFrame([
    ("Total aircraft engines",   f"{df_all['unit number'].nunique()}",  "run-to-failure"),
    ("Total operational cycles", f"{len(df_all):,}",                    "across all engines"),
    ("Min engine lifetime",      f"{life.min()} cycles",                f"engine {life.idxmin()}"),
    ("Max engine lifetime",      f"{life.max()} cycles",                f"engine {life.idxmax()}"),
    ("Mean engine lifetime",     f"{life.mean():.2f} cycles",           f"sd = {life.std(ddof=1):.2f} (sample)"),
    ("RUL range",                f"{df_all['RUL'].min()} - {df_all['RUL'].max()} cycles", "0 = failure point"),
    ("Features used",            f"{X_all.shape[1]}",                   "time + 3 op. settings + 21 sensors"),
], columns=["statistic", "value", "remark"])
save("t01_dataset.csv", t01)

t01_sensors = pd.DataFrame([
    (c.replace("sensor measurement ", ""),
     f"{X_all[c].mean():.2f} +- {X_all[c].std(ddof=1):.2f}",
     f"{X_all[c].min():.2f} - {X_all[c].max():.2f}",
     f"{np.corrcoef(X_all[c], y_all)[0,1]:+.3f}")
    for c in X_all.columns if c.startswith("sensor measurement")
], columns=["sensor", "mean +- sd", "range", "corr with RUL"])
save("t01_sensors.csv", t01_sensors)
t01

---
# Stage 02 · Model performance

Evaluates **both** splits. The row-level split is the conventional protocol and
comparable to prior CMAPSS work; the engine-grouped split puts no engine on both
sides and is the defensible one. Report both.

In [ ]:
rows = []
for s in ("random", "grouped"):
    g, Xtr, Xte, ytr, yte, _ = build_model(s)
    p = g.predict(Xte)
    mse = mean_squared_error(yte, p)
    rows.append((("row-level 80/20" if s == "random" else "engine-level 80/20"),
                 f"{len(Xtr):,}", f"{len(Xte):,}", round(mse, 2),
                 round(np.sqrt(mse), 2), round(mean_absolute_error(yte, p), 2),
                 round(r2_score(yte, p), 4)))
t02 = pd.DataFrame(rows, columns=["split", "n_train", "n_test", "MSE", "RMSE", "MAE", "R2"])
save("t02_model.csv", t02)

# context a reviewer will ask for
base = DummyRegressor().fit(X_train, y_train)
imp = pd.Series(gb.feature_importances_, index=COLS).sort_values(ascending=False)
g2 = GradientBoostingRegressor(random_state=MODEL_SEED).fit(
    X_train.drop(columns=["time"]), y_train)
r2_no_time = r2_score(y_test, g2.predict(X_test.drop(columns=["time"])))

print(f"\n  mean-baseline R2      {r2_score(y_test, base.predict(X_test)):+.4f}")
print(f"  RUL sd on test set    {y_test.std(ddof=1):.2f} cycles")
print(f"  engines in test set   random: {len(set(build_model('random')[5]))}, "
      f"grouped: {len(set(build_model('grouped')[5]))} "
      f"(of {df_all['unit number'].nunique()})")
print(f"\n  'time' is one term of RUL = T_max - time, so check what rests on it:")
print(f"    R2 with 'time'      {t02.loc[t02.split.str.startswith('row'), 'R2'].iloc[0]:.4f}")
print(f"    R2 without 'time'   {r2_no_time:.4f}")
print(f"    importance of time  {imp['time']:.4f}")
t02

---
# Stage 03 · Reference explanations

The single-explanation material for the figures, the local fidelity of each
surrogate, and the aggregated WA-LIME explanation with **both** the magnitudes
(for ranking) and the signed weights (for direction).

In [ ]:
single_rows, agg_rows, meta = [], [], []
pr = Progress(len(INSTANCES), "instances")
for i in INSTANCES:
    row = X_test.iloc[i].values
    pred, actual = float(gb.predict(row.reshape(1, -1))[0]), float(y_test.iloc[i])
    w, fid = lime_weights(row, seed=1000 + i)
    wa_mag, _ = wa_lime_weights(row, seed=2000 + i, signed=False)
    wa_sgn, _ = wa_lime_weights(row, seed=2000 + i, signed=True)

    for r, j in enumerate(np.argsort(-np.abs(w))[:K_DISPLAY], 1):
        single_rows.append((i, r, SHORT(COLS[j]), round(w[j], 4),
                            "raises" if w[j] > 0 else "lowers"))
    for r, j in enumerate(np.argsort(-wa_mag)[:K_DISPLAY], 1):
        agg_rows.append((i, r, SHORT(COLS[j]), round(wa_mag[j], 4),
                         round(wa_sgn[j], 4), "raises" if wa_sgn[j] > 0 else "lowers"))
    meta.append((i, round(pred, 2), actual, round(pred - actual, 2), round(fid, 4)))
    pr.step()
pr.done()

t03_single = save("t03_single_explanation.csv", pd.DataFrame(
    single_rows, columns=["instance", "rank", "feature", "weight", "direction"]))
t03_wa = save("t03_walime_explanation.csv", pd.DataFrame(
    agg_rows, columns=["instance", "rank", "feature", "mean_abs_weight",
                       "mean_signed_weight", "direction"]))
t03_meta = save("t03_instances.csv", pd.DataFrame(
    meta, columns=["instance", "predicted_RUL", "actual_RUL", "residual", "R2_local"]))
print(f"\nmean local fidelity: {t03_meta.R2_local.mean():.4f}")
t03_meta

In [ ]:
# the explanation for the first instance, as it would appear in a figure
i0 = INSTANCES[0]
t03_single[t03_single.instance == i0]

---
# Stage 04 · Explanation stability — the headline result

Three things a 5-run point estimate cannot give:

1. **Jaccard and Spearman over R repetitions per instance, pooled across
   instances**, with a bootstrap interval resampled over *runs* (pairs are
   dependent, so a naive interval over pairs is too narrow).
2. **The B-vs-stability curve**, so the aggregation depth is a reported choice
   rather than an arbitrary one.
3. **A compute-matched control.** WA-LIME with B=5 spends 5× the perturbation
   budget of one LIME call, so without plain LIME at `B × num_samples` a
   reviewer cannot tell aggregation from extra sampling. This is the first
   baseline they will ask for.

This is the long cell — with `QUICK = False` expect roughly 40 minutes.

In [ ]:
METHODS = [("LIME", lambda row, s: lime_weights(row, s)[0], 1)]
for b in B_SWEEP:
    METHODS.append((f"WA-LIME B={b}",
                    (lambda b_: lambda row, s: wa_lime_weights(row, s, B=b_)[0])(b), b))
if RUN_COMPUTE_MATCHED:
    METHODS.append((f"LIME n={B_AGG*N_PERTURB} (compute-matched to B={B_AGG})",
                    lambda row, s: lime_weights(row, s, n_perturb=B_AGG*N_PERTURB)[0],
                    B_AGG))

rows, per_inst, detail = [], [], {}
pr = Progress(len(METHODS) * len(INSTANCES), "method x instance")
for name, fn, cost in METHODS:
    pooled = {m: [] for m in [f"jaccard@{k}" for k in K_LIST] + ["spearman"]}
    boots = []
    for i in INSTANCES:
        row = X_test.iloc[i].values
        W = np.array([fn(row, 7000 + i * 100 + r) for r in range(R)])
        st = pairwise_stability(W)
        for m in pooled:
            pooled[m].append(st[m])
        boots.append(bootstrap_over_runs(W, K_DISPLAY, seed=i))
        per_inst.append((name, i, *[round(st[f"jaccard@{k}"], 4) for k in K_LIST],
                         round(st["spearman"], 4)))
        pr.step()
    means = {m: float(np.mean(v)) for m, v in pooled.items()}
    sds = {m: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for m, v in pooled.items()}
    lo, hi = float(np.mean([b[0] for b in boots])), float(np.mean([b[1] for b in boots]))
    detail[name] = dict(mean=means, sd=sds, boot95=[lo, hi],
                        perturbations=cost * N_PERTURB)
    rows.append((name, f"{cost*N_PERTURB:,}",
                 *[f"{means[f'jaccard@{k}']:.4f} ({sds[f'jaccard@{k}']:.4f})" for k in K_LIST],
                 f"{means['spearman']:.4f} ({sds['spearman']:.4f})",
                 f"[{lo:.3f}, {hi:.3f}]"))
pr.done()

t04 = save("t04_stability.csv", pd.DataFrame(rows, columns=(
    ["method", "perturbations"] + [f"Jaccard@{k} (sd)" for k in K_LIST] +
    ["Spearman (sd)", f"95pct boot CI on Jaccard@{K_DISPLAY}"])))
save("t04_stability_per_instance.csv", pd.DataFrame(
    per_inst, columns=["method", "instance"] + [f"jaccard@{k}" for k in K_LIST] + ["spearman"]))
t04

### The compute-matched verdict

**Read this before writing the abstract.** If WA-LIME does not beat plain LIME
at equal budget, the claim "aggregation stabilises explanations" is not supported
as stated. The honest options are to re-frame the contribution (simplicity, or
the budget/stability curve above), or to find a regime where aggregation
genuinely wins — the candidate is robustness to *background* choice, which
raising `num_samples` cannot address.

Do not drop this row from the results table because it is inconvenient. Its
absence is more conspicuous than its presence.

In [ ]:
if RUN_COMPUTE_MATCHED:
    k = K_DISPLAY
    wa = detail[f"WA-LIME B={B_AGG}"]["mean"]
    cm = detail[f"LIME n={B_AGG*N_PERTURB} (compute-matched to B={B_AGG})"]["mean"]
    dj, ds = wa[f"jaccard@{k}"] - cm[f"jaccard@{k}"], wa["spearman"] - cm["spearman"]
    print(f"at {B_AGG*N_PERTURB:,} perturbations:")
    print(f"  WA-LIME B={B_AGG}            Jaccard@{k} = {wa[f'jaccard@{k}']:.4f}   "
          f"Spearman = {wa['spearman']:.4f}")
    print(f"  plain LIME, same budget   Jaccard@{k} = {cm[f'jaccard@{k}']:.4f}   "
          f"Spearman = {cm['spearman']:.4f}")
    print(f"  difference (WA - LIME)    dJ = {dj:+.4f}   drho = {ds:+.4f}\n")
    if dj <= 0 and ds <= 0:
        print("WA-LIME does NOT beat the compute-matched baseline on either metric.")
        print("Do not claim aggregation is the source of the gain -- re-frame, or")
        print("find a regime where aggregation wins (background robustness).")
    elif dj > 0 and ds > 0:
        print("WA-LIME beats the compute-matched baseline on BOTH metrics.")
        print("This is the result to lead with -- quote both rows together.")
    else:
        print("Mixed: WA-LIME wins on one metric, loses on the other. Report both,")
        print("and do not claim a clean win on stability.")

---
# Stage 05 · Rank occupancy and signal-to-noise

Rank occupancy is a sharper claim than an averaged Jaccard: it shows **where**
instability lives and where aggregation helps. The SNR table measures the
mechanism rather than asserting it — if each run is a noisy estimate
β̂ = β* + ε, rank stability should track |mean w| / sd(w).

**On `R`.** It sets how precisely each occupancy *proportion* is estimated,
and the precision is binomial: SE ≈ √(p(1−p)/R), which does **not** involve the
number of features. At R=25 the modal share for rank 8 carries an SE near 10
percentage points; at R=100 it is about 4. Running R past the 25 candidate
features is therefore not redundant — the goal is not to enumerate the possible
occupants but to estimate how often each one wins.

**Caveat on `n_distinct`.** The count of distinct occupants is *not* a stable
statistic: it grows with R (4.3 at R=10, 6.2 at 25, 7.6 at 50, 8.5 at 100, 10.0
at 400, measured on one instance). It is a species-richness count, so more draws
keep revealing rarer occupants. **Only compare `n_distinct` between methods run
at the same R**, and always report R next to it. Modal share is the statistic
that converges; treat `n_distinct` as descriptive.

In [ ]:
occ, occ_full, snr = [], [], []
pr = Progress(len(INSTANCES), "instances")
for i in INSTANCES:
    row = X_test.iloc[i].values
    Wl = np.array([lime_weights(row, 31000 + i * 100 + r)[0] for r in range(R)])
    Ww = np.array([wa_lime_weights(row, 41000 + i * 100 + r)[0] for r in range(R)])

    for label, W in (("LIME", Wl), (f"WA-LIME B={B_AGG}", Ww)):
        order = np.argsort(-np.abs(W), axis=1)
        for r in range(K_DISPLAY):
            c = Counter(order[:, r]); tot = sum(c.values()); top = c.most_common()
            occ.append((label, i, r + 1, SHORT(COLS[top[0][0]]),
                        round(top[0][1] / tot, 4), len(c),
                        round(float(np.mean(np.abs(W)[np.arange(len(W)), order[:, r]])), 4)))
            for feat, n in top:
                if n / tot >= 0.02:
                    occ_full.append((label, i, r + 1, SHORT(COLS[feat]), round(n / tot, 4)))

    mu, sd = Wl.mean(0), Wl.std(0, ddof=1)
    ranks = np.argsort(np.argsort(-np.abs(Wl), axis=1), axis=1) + 1
    for j in np.argsort(-np.abs(mu)):
        snr.append((i, SHORT(COLS[j]), round(mu[j], 4), round(sd[j], 4),
                    round(abs(mu[j]) / sd[j], 2) if sd[j] else np.inf,
                    round(ranks[:, j].mean(), 2), round(ranks[:, j].std(ddof=1), 2),
                    round(float((ranks[:, j] <= K_DISPLAY).mean()), 4)))
    pr.step()
pr.done()

t05_occ = save("t05_rank_occupancy.csv", pd.DataFrame(occ, columns=[
    "method", "instance", "rank", "modal_feature", "modal_share",
    "n_distinct", "mean_abs_weight"]))
save("t05_rank_occupancy_full.csv", pd.DataFrame(
    occ_full, columns=["method", "instance", "rank", "feature", "share"]))
t05_snr = save("t05_feature_snr.csv", pd.DataFrame(snr, columns=[
    "instance", "feature", "mean_weight", "sd_weight", "snr",
    "mean_rank", "rank_sd", f"top{K_DISPLAY}_rate"]))

# pooled across instances -- this is the table for the paper
pooled = (t05_occ.groupby(["method", "rank"])[["modal_share", "n_distinct"]]
          .mean().round(4).reset_index()
          .pivot(index="rank", columns="method",
                 values=["modal_share", "n_distinct"]))
save("t05_rank_pooled.csv", pooled.reset_index())
pooled

In [ ]:
# per-feature SNR, averaged over instances -- the mechanism table
snr_pooled = (t05_snr.groupby("feature")
              .agg(mean_abs_weight=("mean_weight", lambda s: np.mean(np.abs(s))),
                   noise_sd=("sd_weight", "mean"),
                   snr=("snr", "mean"),
                   mean_rank=("mean_rank", "mean"),
                   rank_sd=("rank_sd", "mean"),
                   top8_rate=(f"top{K_DISPLAY}_rate", "mean"))
              .sort_values("mean_abs_weight", ascending=False).round(3))
save("t05_feature_snr_pooled.csv", snr_pooled.reset_index())
print("noise floor across features: "
      f"{snr_pooled.noise_sd.min():.3f} to {snr_pooled.noise_sd.max():.3f}")
print("(a roughly constant sd regardless of coefficient size is the "
      "homoscedastic epsilon the variance-reduction argument assumes)")
snr_pooled

---
# Summary

Everything written this run. Copy the CSVs straight into the manuscript tables.

In [ ]:
import glob
print(f"config: QUICK={QUICK}  split={SPLIT}  instances={len(INSTANCES)}  "
      f"R={R}  B_agg={B_AGG}\n")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv"))):
    print(f"  {os.path.basename(p):<38} {sum(1 for _ in open(p))-1:>5} rows")
if QUICK:
    print("\n*** QUICK=True -- these are smoke-test numbers, NOT for the paper. ***")
    print("*** Set QUICK = False in the config cell and re-run for the real ones. ***")